In [ ]:
import os
import json

# 사용자 설정
inference_dir = "./raw_inference/amboss"  # JSONL 파일들이 있는 디렉토리
reference_file = "./resources/processed_gpt4_responses_amboss_fixed.json"

# gold_report → id 매핑 만들기
with open(reference_file, "r", encoding="utf-8") as f:
    reference_data = json.load(f)
    report_to_id = {entry["gold_report"]: entry["id"] for entry in reference_data}

# inference_dir 내부 파일들 in-place 수정
for filename in os.listdir(inference_dir):
    if not filename.endswith(".jsonl"):
        continue

    file_path = os.path.join(inference_dir, filename)

    print(f"🔧 Processing {filename}...")

    # 파일 전체를 메모리에 읽기
    new_lines = []
    matched_count = 0

    with open(file_path, "r", encoding="utf-8") as fin:
        for line in fin:
            try:
                entry = json.loads(line)
                caption = entry.get("gold_caption", "").strip()

                if caption in report_to_id:
                    entry["id"] = report_to_id[caption]
                    matched_count += 1
                else:
                    entry["id"] = None  # 매칭 실패 시 None

                new_lines.append(json.dumps(entry, ensure_ascii=False))
            except Exception as e:
                print(f"❗ Error reading line in {filename}: {e}")
                new_lines.append(line)

    # 덮어쓰기
    with open(file_path, "w", encoding="utf-8") as fout:
        for l in new_lines:
            fout.write(l + "\n")

    print(f"✅ {filename}: {matched_count}개 항목에 id 추가 (in-place overwrite)")
